In [1]:
import time

TS = time.time()

def ticks_us():

    """for testing - microseconds since we started"""

    return int((time.time() - TS) * 1E6)

def ticks_diff(a, b):

    """So it looks like the time library we are using"""
    
    return a - b



In [2]:
a = ticks_us()

In [3]:
b = ticks_us()
ticks_diff(b, a)

1085560

In [5]:
# expo functions - default is to decay to 10% after 1 second

# y = 1/(10^t) decays to 0.1 in 1s, and then 0.01 in 2s

from array import array
import math

def build_expo_array(npoints=100):

    """build a lookup table for y=1/(10**x)"""
    
    arr = array("H", [])  # using H (unsigned short) - all envelopes and LFOs will be positive modulation values
    step_size = 2.0/npoints
    t = 0
    while t < 2.0:
        v = 1/(10**t)
        vint = int(v * 65535) - 1  # convert our frac from 0 to 1 to cover the max range of our 16 bit integer
        arr.append(vint)
        t += step_size

    return arr

def build_saw_array(npoints=100):

    arr = array("H", [])  # unsigned short
    step_size = 65535//npoints
    v = 0
    index = 0
    for _ in range(npoints):
        arr.append(v)
        v += step_size
        
    return arr

def build_ramp_array(saw_array, npoints=100):

    arr = array("H", [])  # unsigned short
    for v in saw_array:
        arr.append(65535 - v)
    
    return arr

def build_triangle_array(saw_array, npoints=100):

    arr = array("H", [])  # unsigned short
    for v in saw_array[::2]:
        arr.append(v)
    # now walk backwards thru the array, mirroring it
    index = len(arr) - 1
    while len(arr) < npoints:
        arr.append(arr[index])
        index -= 1

    return arr

def build_sine_array(npoints=100):

    arr = array("H", [])  # unsigned short
    increment = 2 * math.pi / npoints
    i = 0
    while len(arr) < npoints:
        f = math.sin(i)
        arr.append(int((f + 1) * 32767))
        i += increment
    return arr

def build_sharkfin_array(expo_array, npoints=100):

    arr = array("H", [])  # unsigned short
    for v in expo_array[::2]:
        arr.append(v)
    # now walk backwards thru the array
    for v in expo_array[::2]:
        arr.append(65535 - v)

    return arr
    

    
    

q = build_saw_array()
for e, q in enumerate(build_sharkfin_array(build_expo_array(npoints=100))):
    print(f"{e}\t{q}")
    
    
    

0	65534
1	59767
2	54508
3	49712
4	45338
5	41348
6	37710
7	34392
8	31366
9	28606
10	26088
11	23793
12	21699
13	19790
14	18048
15	16460
16	15012
17	13691
18	12486
19	11387
20	10385
21	9471
22	8638
23	7878
24	7184
25	6552
26	5975
27	5449
28	4970
29	4532
30	4133
31	3770
32	3438
33	3135
34	2859
35	2607
36	2378
37	2169
38	1978
39	1803
40	1645
41	1500
42	1368
43	1247
44	1137
45	1037
46	946
47	862
48	786
49	717
50	1
51	5768
52	11027
53	15823
54	20197
55	24187
56	27825
57	31143
58	34169
59	36929
60	39447
61	41742
62	43836
63	45745
64	47487
65	49075
66	50523
67	51844
68	53049
69	54148
70	55150
71	56064
72	56897
73	57657
74	58351
75	58983
76	59560
77	60086
78	60565
79	61003
80	61402
81	61765
82	62097
83	62400
84	62676
85	62928
86	63157
87	63366
88	63557
89	63732
90	63890
91	64035
92	64167
93	64288
94	64398
95	64498
96	64589
97	64673
98	64749
99	64818


In [46]:
ARR = build_expo_array()
SAW = build_saw_array()
RAMP = build_ramp_array(SAW)
TRI = build_triangle_array(SAW)
SINE = build_sine_array()
SHARK = build_sharkfin_array(ARR)

class LFO:

    def __init__(self, saw, ramp, tri, sine, shark):

        """set up references to all of the various function lookup tables"""

        #  TODO: modulatable depth and rate
        
        self.saw = saw
        self.ramp = ramp
        self.tri = tri
        self.sine = sine
        self.shark = shark

        self.shapes = [self.saw, self.ramp, self.tri, self.sine, self.shark]  # so we can refer to them by index
        self.shape = self.saw  # default
        self.array_length = len(self.shape)  # NOTE - assumes all wavetables have the same resolution

        self._rate = 32768  # default value - internally we use 16 bit rather than 8 for finer grained resolution
        self.set_divisor()

        self.last = ticks_us()  # what was the last time we got a value? use this to determine how far to progress the array index
        self.current_index = 0  # record the level we sent last so we can smoothly change parameters without causing a stutter in the output

    @property
    def rate(self):

        return self._rate >> 8

    @rate.setter
    def rate(self, new_value):

        self._rate = new_value << 8
        self.set_divisor()  # calculate the new divisor to get the array index

    def set_shape(self, ind):

        self.shape = self.shapes[ind]

    def set_divisor(self):

        """Given a value from 0-255 corresponding to 10 Hz to 0.1 Hz, calculate how many microseconds between positions in the array.
        """

        interval = ((65536 - self._rate) / 65535.0 * 1E7) / self.array_length
        # dividing up to 10 seconds across the array indices
        self.divisor = int(interval)
    
    def get(self, caller=None):

        """caller is optional and allows us to specify a unique phase offset depending on who is asking for an LFO value"""

        # TODO: phase offset

        timenow = ticks_us()
        tdelta = ticks_diff(timenow, self.last)
        self.last = timenow
        
        index_increment = tdelta // self.divisor
        self.current_index += index_increment
        if self.current_index >= self.array_length:
            self.current_index = self.current_index - self.array_length  # wrap around
        return self.shape[self.current_index]







mylfo = LFO(SAW, RAMP, TRI, SINE, SHARK)
mylfo.rate = 220
mylfo.set_shape(3)



for x in range(100):
    print(f"{x}\t{mylfo.get()}")
    time.sleep(0.1)
    if x > 50:
        mylfo.rate = 100


0	32767
1	46718
2	58014
3	64504
4	64953
5	59276
6	48552
7	34824
8	20704
9	8880
10	1603
11	258
12	5100
13	15209
14	28660
15	42892
16	55197
17	63232
18	65469
19	61480
20	52026
21	38906
22	24618
23	11880
24	3118
25	0
26	3118
27	11880
28	24618
29	38906
30	52026
31	61480
32	65469
33	63232
34	55197
35	42892
36	28660
37	15209
38	5100
39	258
40	1603
41	8880
42	20704
43	34824
44	48552
45	59276
46	64953
47	64504
48	58014
49	46718
50	32766
51	18815
52	16981
53	15209
54	13507
55	11880
56	10336
57	8880
58	7519
59	6257
60	5100
61	4053
62	3118
63	2301
64	1603
65	1029
66	580
67	258
68	64
69	0
70	64
71	258
72	580
73	1029
74	1603
75	2301
76	3118
77	4053
78	5100
79	6257
80	7519
81	8880
82	10336
83	11880
84	13507
85	15209
86	16981
87	18815
88	20704
89	22641
90	24618
91	26627
92	28660
93	30709
94	32767
95	34824
96	36873
97	38906
98	40915
99	42892


In [42]:
mylfo._rate

51200

In [47]:
import time

def fpmult(a, b):

    """"Scale a down by a factor b of 0-65536 (returns a * (b/65536))"""
    
    c = a * b
    return c >> 16
  

        
    

class ADSR:

    def __init__(self, arr):

        """Minimum interesting phase time = 10 ms, maximum = 10 s (3 orders of magnitude)
        This object expects parameters in the range of 0..255 so 255 = a 10s attack, decay or release
        function is (blah/255) * 3 + 4  -> maps an input of 0..255 to a range from 4 to 7, which is orders of mag of microseconds
        then need to spread the lookup table across that whole range.

        arr is the array generated by the build_expo_array function which describes an exponential decay
        """

        self.a = 127  # sensible defaults
        self.d = 127
        self.s = 127
        self.r = 127
        self.array_length = len(arr)  # shouldn't really change but we might use higher/lower resolution lookup tables
        # need this to determine when to transition from a to d or d to s
        self.arr = arr  # our reference to the global expo array

        self.refresh_settings()
        
        self.gate_starts = {}  # a dictionary of caller: start time in microseconds
        self.gate_status = {}  # caller: True/False - gate signal. True = gate is open
        self.level = 0  # current output, 0-32767
        self.sustaining = False
        self.decaying = False
        self.attacking = False
        self.releasing = False
        self.releasing_from = None  # we can't always assume we are releasing from the sustain level if note was released during decay phase
        self.attack_fastforward = 0  # array offset if we want to start attacking from a higher base level

    def get_fastforward(self):

        ind = -1
        l = 0
        arrlength = len(self.arr)
        while l < self.level and ind < arrlength:
            ind += 1
            l = 65535 - self.arr[ind]
        return ind       

    def refresh_settings(self):

        # todo: use setter methods
        
        self.a_d = self.get_divisor(self.a)
        self.d_d = self.get_divisor(self.d)
        self.r_d = self.get_divisor(self.r)
        #self.s = self.s << 7

    def gate(self, caller, status):

        """As far as caller is concerned, the ADSR has been running since the start of the gate function. Status = True either starts or restarts
        the envelope from the beginning of the attack phase. As long as gate is open we will hold at the sustain value after going thru attack
        and decay phases. When status=False, the decay phase commences."""

        self.gate_starts[caller] = ticks_us()
        self.gate_status[caller] = status

        # if we recieved gate on or off, then we aren't in either of these states:
        self.decaying = False
        self.sustaining = False
        self.releasing_from = self.level  # store the level to use for vertically scaling the next phase
        
        if status:
            self.attacking = True
            self.releasing = False
            self.attack_fastforward = self.get_fastforward()
        else:         
            self.attacking = False
            self.releasing = True

    def get_divisor(self, rate):

        """Given a value from 0-255 corresponding to 1ms to 10s, calculate how many microseconds between positions in the array.
        For the default 100 point array to be covered in 2 seconds, each increment is 20,000 us"""

        length = 10**((rate/255.0) * 3 + 4)  # map 0-255 to 10 - 10,000 ms
        interval = length/(len(self.arr)/2)  # table covers 2 "intervals" - 100 points for a 2-second sample so 50 points per second

        return int(interval)


    def get(self, caller):

        """return the current magnitude of the envelope signal. We add timedelta to the
        start of the gate signal to determine how far thru the cycle we are."""

        # TODO: linear interpolation between coarser table values
        
        if self.sustaining:
            return self.level

        tdelta = ticks_diff(ticks_us(), self.gate_starts[caller])
            
        if self.gate_status[caller]:
            # gate is true but we didn't reach the sustain level, must be attacking or decaying
            if self.attacking:
                index = tdelta // self.a_d + self.attack_fastforward

                # we need to "fast-forward" thru the attack array if we are coming from some residual decay/release level
                
                #print(ticks_us(), self.gate_starts[caller])
                #print(self.a_d)
                #print("atack and index is", index)
                if index < self.array_length and self.level < 65535:
                    # within attack phase
                    lev = 65535 - self.arr[index] # exponential INCREASE, so 1 minus
                    # add the level if we are re-initiating an attack during the decay phase
                    self.level = lev
                    return min(lev, 65535)  # this is the only place we could go over 255 because of adding the previous level. Never want to do this.
                    # level >= 255 will be detected in the next "get" call and push us into the decay phase.
                else:  # finished the attack array and now it is time to decay
                    self.attacking = False
                    self.decaying = True
                    self.gate_starts[caller] = ticks_us()  # reset time counter for decay curve now
                    lev = 65535 - self.arr[-1]
                    self.level = lev
                    self.releasing_from = lev  # record how high we got and use it to scale the decay phase
                    return lev
            else:
                # we must be decaying, sustain was handled further up
                index = tdelta // self.d_d
                if index < self.array_length:
                    decay_point = self.arr[index]  # those arrays follow each other
                    # we need to vertically scale the decay to squash it into the space between max level and sustain level
                    # then add the "floor" of the sustain level
                    scalerange = self.releasing_from - self.s
                    lev = self.s + fpmult(decay_point, scalerange)
                    #lev = self.level - fpmult(decay_point, (self.releasing_from << 1))
                    
                    self.level = lev
                    return lev
                else:  # we reached the end of the decay table so now just hold at the sustain level
                    self.decaying = False
                    self.sustaining = True
                    return self.level  # always need to return something so start sustaining
                    # note that we never actually reach the value specified by the sustain parameter because the expo function can never reach it
                    # we will always be just a little over, but this avoids a discontinuity when exiting the decay function
        else:  # release phase
            if self.releasing:
                index = tdelta // self.r_d
                if index < self.array_length:
                    release_point = self.arr[index]
                    lev = fpmult(release_point, self.releasing_from)  # need to vertically squash release values to decrease from the current level
                    # releasing_from was set when the gate signal went False
                    self.level = lev
                    return lev
                else:
                    self.releasing = False
                    return 0  # we ran past the end of the release table, envelope has completed.
            else:
                return 0  # not sustaining, not gated, not releasing, nothing is happening
             
        

    

In [7]:
ARR = build_expo_array(npoints=500)



In [188]:
10**((1/255.0) * 3 + 4)

10274.5948544618

In [189]:
10**((255/255.0) * 3 + 4)

10000000.0

In [48]:
event_times = [20, 30, 60, 100,]

state = True
ENV = ADSR(ARR)
ENV.gate("A", state)
cnt = 0
ENV.a = 60
ENV.d = 120
ENV.s = 16500
ENV.r = 10
ENV.refresh_settings()
while cnt < 200:
    time.sleep(0.01)
    print(ENV.get("A"))
    cnt += 1
    if len(event_times) == 0:
        continue
    if cnt > event_times[0]:
        event_times.pop(0)
        ENV.gate("A", state)
        state = not state

22238
39447
49075
55150
58983
61402
62928
63890
64498
64850
60593
58609
53175
49948
47005
44321
41873
39640
37604
36654
34880
46636
53611
58012
60789
62541
63646
64344
64818
64850
60593
10528
1667
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
26048
40621
49816
55617
59560
61765
63157
64035
64589
64850
62672
58609
54904
49948
47005
44321
41873
39640
37604
35746
34053
32508
31100
29815
28643
27575
26600
25711
24900
24161
23487
22872
22312
21800
21333
20908
20520
20166
19843
19549
3097
490
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [159]:
ARR[-1]

328

In [160]:
ARR

array('h', [32767, 32616, 32466, 32317, 32168, 32021, 31873, 31727, 31581, 31436, 31292, 31148, 31005, 30862, 30721, 30579, 30439, 30299, 30160, 30021, 29883, 29746, 29609, 29473, 29338, 29203, 29069, 28935, 28802, 28670, 28538, 28407, 28277, 28147, 28017, 27889, 27760, 27633, 27506, 27380, 27254, 27129, 27004, 26880, 26756, 26633, 26511, 26389, 26268, 26147, 26027, 25907, 25788, 25670, 25552, 25435, 25318, 25201, 25086, 24970, 24856, 24741, 24628, 24515, 24402, 24290, 24178, 24067, 23956, 23846, 23737, 23628, 23519, 23411, 23304, 23196, 23090, 22984, 22878, 22773, 22668, 22564, 22461, 22357, 22255, 22152, 22051, 21949, 21848, 21748, 21648, 21549, 21450, 21351, 21253, 21155, 21058, 20961, 20865, 20769, 20674, 20579, 20484, 20390, 20296, 20203, 20110, 20018, 19926, 19834, 19743, 19652, 19562, 19472, 19383, 19294, 19205, 19117, 19029, 18942, 18855, 18768, 18682, 18596, 18510, 18425, 18341, 18256, 18172, 18089, 18006, 17923, 17841, 17759, 17677, 17596, 17515, 17435, 17355, 17275, 17195, 1

In [167]:
0 < 0

False

In [57]:
from array import array

ARR = array("H", range(50,100))

class MyThing:

    def __init__(self, array_reference):

        self.array = array_reference
        self._a = 10

    def get_value(self, index):

        return self.array[index]

    @property
    def a(self):
        
        return self._a
    
    @a.setter
    def a(self, new_value):

        self._a = new_value
        self.side_effect()  # calculate the new divisor to get the array index

    def side_effect(self):

        print("curse you, red baron!")

baz = MyThing(ARR)

In [58]:
baz.a

10

In [71]:
baz.a = 15

curse you, red baron!


In [73]:
baz.__setattr__("a", 100)

curse you, red baron!


In [74]:
baz.a

100

In [32]:
mylfo = LFO(SAW, RAMP, TRI, SINE, SHARK)
mylfo.rate = 250
mylfo.set_shape(3)
mylfo.update_settings()

event_times = [1200, 1800]

state = True
ENV = ADSR(ARR)
ENV.gate("A", state)
cnt = 0
ENV.a = 250
ENV.d = 120
ENV.s = 16500
ENV.r = 10
ENV.refresh_settings()
while cnt < 2000:
    time.sleep(0.01)
    evalue = ENV.get("A")
    lvalue = mylfo.get()
    print(f"{cnt}\t{evalue}\t{lvalue}\t{evalue * lvalue >> 16}")
    cnt += 1
    if len(event_times) == 0:
        continue
    if cnt > event_times[0]:
        event_times.pop(0)
        ENV.gate("A", state)
        state = not state





0	64850	40915	40486
1	64850	48552	48043
2	64850	56653	56059
3	64850	61480	60836
4	64850	64953	64273
5	64850	65469	64783
6	64850	63930	63260
7	64850	59276	58655
8	64850	53653	53091
9	64850	46718	46228
10	64850	36873	36487
11	64850	28660	28360
12	64850	18815	18618
13	64850	11880	11755
14	64850	5100	5046
15	64850	1603	1586
16	1	0	0
17	1	1029	0
18	1	5100	0
19	1	10336	0
20	1	16981	0
21	1	26627	0
22	1	32767	0
23	1	40915	0
24	1	48552	0
25	1	56653	0
26	1	61480	0
27	1	64953	0
28	1	65469	0
29	1	63930	0
30	1	59276	0
31	1	53653	0
32	1	46718	0
33	2951	36873	1660
34	2951	28660	1290
35	2951	18815	847
36	2951	11880	534
37	2951	6257	281
38	2951	1603	72
39	2951	64	2
40	2951	1029	46
41	2951	4053	182
42	2951	10336	465
43	2951	16981	764
44	2951	26627	1198
45	2951	32767	1475
46	2951	40915	1842
47	2951	48552	2186
48	2951	56653	2551
49	2951	61480	2768
50	5768	64953	5716
51	5768	65469	5762
52	5768	63930	5626
53	5768	59276	5217
54	5768	53653	4722
55	5768	46718	4111
56	5768	36873	3245
57	5768	28660	2522
58	5768	

In [29]:
2124939950 >> 16

32424